### ライブラリの読み込み

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

### データの読み込み

In [2]:
# TCVファイルの読み込み
train_A = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
train_B = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
train_C = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
train_D = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test_data = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 学習データの結合
train_data = pd.concat([train_A, train_B, train_C, train_D], ignore_index=True)

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

### 特徴量エンジニアリング

In [5]:
# タイムスタンプをdatetime型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])

In [6]:
# 最新の行動からの経過時間（秒）
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [7]:
# ユーザーごとの行動回数集計
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# 商品ごとの人気度
product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [9]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")

### XGBoost 用データセットの準備 ～ モデル学習 ～ nGCDの計算

In [10]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users"
]

In [11]:
# クラス不均衡に対する学習データのバランス調整
train_data_pos = train_data[train_data["relevance"] >= 2]  # 広告クリックと購入のデータ
train_data_neg = train_data[train_data["relevance"] < 2]   # その他のデータ

In [12]:
# 少数派クラス（購入・広告クリック）をオーバーサンプリング
train_data_pos_oversampled = resample(train_data_pos, 
                                      replace=True, 
                                      n_samples=len(train_data_neg),  # 多数派クラスのサンプル数に合わせる
                                      random_state=42)

In [13]:
# サンプリング後のデータ
train_data_balanced = pd.concat([train_data_neg, train_data_pos_oversampled])

In [14]:
# クロスバリデーション設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [15]:
# 評価指標を格納するリスト
ndcg_scores = []

In [16]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]
    
    # 特徴量とラベル
    X_train = train_set[features]
    y_train = train_set["relevance"]
    X_val = val_set[features]
    y_val = val_set["relevance"]
    
    # クエリごとのデータ数（ユーザーごと）
    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()
    
    # XGBoost DMatrix に変換
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)

    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    # 学習パラメータ設定
    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.1,
        "max_depth": 6,
        "min_child_weight": 10,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "gamma": 0.1
    }
    
    # モデル学習
    model = xgb.train(
        params, dtrain, num_boost_round=200,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=20, verbose_eval=10
    )
    
    # バリデーションデータの nDCG 計算
    ground_truth_val = defaultdict(dict)
    for _, row in val_set.iterrows():
        ground_truth_val[row["user_id"]][row["product_id"]] = row["relevance"]

    # バリデーションデータのコピーを作成
    val_set_copy = val_set.copy()

    # バリデーションデータに対してスコアを設定
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    # nDCG の計算
    def dcg_at_k(relevance_scores, k):
        relevance_scores = np.asarray(relevance_scores)[:k]
        return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))

    def ndcg_at_k(relevance_scores, k):
        dcg = dcg_at_k(relevance_scores, k)
        ideal_dcg = dcg_at_k(sorted(relevance_scores, reverse=True), k)
        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    def evaluate_ndcg(data, ground_truth, k=22):
        ndcg_scores = []
        for user_id, group in data.groupby("user_id"):
            predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
            relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
            ndcg_scores.append(ndcg_at_k(relevance_scores, k))
        return np.mean(ndcg_scores) if ndcg_scores else 0
    
    ndcg_val = evaluate_ndcg(val_set_copy, ground_truth_val, k=22)
    ndcg_scores.append(ndcg_val)
    print(f"Validation nDCG@22: {ndcg_val:.4f}")

[0]	train-ndcg:0.89141	val-ndcg:0.92731
[10]	train-ndcg:0.89896	val-ndcg:0.93079
[20]	train-ndcg:0.90042	val-ndcg:0.93168
[30]	train-ndcg:0.90201	val-ndcg:0.93245
[40]	train-ndcg:0.90398	val-ndcg:0.93331
[50]	train-ndcg:0.90488	val-ndcg:0.93380
[60]	train-ndcg:0.90567	val-ndcg:0.93436
[70]	train-ndcg:0.90682	val-ndcg:0.93498
[80]	train-ndcg:0.90778	val-ndcg:0.93550
[90]	train-ndcg:0.90841	val-ndcg:0.93591
[100]	train-ndcg:0.90907	val-ndcg:0.93632
[110]	train-ndcg:0.90983	val-ndcg:0.93675
[120]	train-ndcg:0.91071	val-ndcg:0.93724
[130]	train-ndcg:0.91141	val-ndcg:0.93764
[140]	train-ndcg:0.91215	val-ndcg:0.93816
[150]	train-ndcg:0.91279	val-ndcg:0.93855
[160]	train-ndcg:0.91320	val-ndcg:0.93876
[170]	train-ndcg:0.91391	val-ndcg:0.93914
[180]	train-ndcg:0.91443	val-ndcg:0.93948
[190]	train-ndcg:0.91489	val-ndcg:0.93979
[199]	train-ndcg:0.91531	val-ndcg:0.94001
Validation nDCG@22: 0.7494
[0]	train-ndcg:0.89099	val-ndcg:0.92736
[10]	train-ndcg:0.89883	val-ndcg:0.93059
[20]	train-ndcg:0.900

In [17]:
# クロスバリデーションの平均 nDCG を表示
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.7492


### 予測と提出データの作成

In [18]:
# 推薦候補商品リストを作成
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [19]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [20]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")

In [21]:
# 欠損値処理（初めてのユーザーや商品に対して）
test_data.fillna(0, inplace=True)

In [22]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [23]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [24]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [25]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.1570


In [26]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]

In [27]:
# rankを整数に変換
submission['rank'] = submission['rank'].astype(int)

In [28]:
# 保存
submission.to_csv('./predict_file/prediction.tsv', sep="\t", index=False, header=False)